# 02 · Cómo se construye el clima de una ciudad

Este notebook explica el **método**: cómo se pasa de series de nodo a series de
ciudad, y qué decisiones cambian los números. Si solo quieres pedir datos, ve al
**03 · consultas**.

> **Este notebook es autocontenido.** Corre de arriba a abajo sin necesitar los
> otros. La primera celda es el único requisito.

## La cadena

```
nodo-hora   --(media ponderada por `peso`)-->   ciudad-hora
ciudad-hora --(media / mínimo / máximo)     -->  ciudad-día
```

## Requisitos

1. Los CSV de pesos: `python -m urbano.construir`
2. Los parquet anuales en `Data/Tamaulipas/<año>/Finales/completo/`

In [1]:
# --- arranque (correr siempre primero) ---------------------------------------
# Localiza la raíz del proyecto subiendo por el árbol de directorios, para que el
# notebook funcione sin importar desde dónde se lance Jupyter.
#
# La trampa: ESTA carpeta se llama `notebooks/urbano`. Buscar un directorio
# llamado "urbano" la encuentra a ella, y Python la importa como paquete de
# espacio de nombres vacío -> "cannot import name 'config' from 'urbano'
# (unknown location)". Por eso se exige el `__init__.py`: solo el paquete real
# lo tiene.
import os, sys


def _raiz_del_proyecto(inicio="."):
    d = os.path.abspath(inicio)
    while not os.path.isfile(os.path.join(d, "urbano", "__init__.py")):
        padre = os.path.dirname(d)
        if padre == d:
            raise RuntimeError(
                f"No encuentro el paquete `urbano` subiendo desde "
                f"{os.path.abspath(inicio)}. Abre Jupyter dentro del repo.")
        d = padre
    return d


RAIZ = _raiz_del_proyecto()
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)

# Si una celda anterior ya importó el `urbano` equivocado, quedó cacheado y el
# arreglo del path no bastaría: hay que descartarlo para no exigir reiniciar el
# kernel.
for _m in [m for m in list(sys.modules) if m == "urbano" or m.startswith("urbano.")]:
    del sys.modules[_m]

import urbano
print("raíz del proyecto:", RAIZ)
print("paquete urbano   :", os.path.dirname(urbano.__file__))

# El paquete `urbano` se edita mientras el notebook está abierto, y Python NO
# recarga un módulo ya importado: la celda seguiría usando la versión vieja y
# fallaría con errores del tipo "['cve_mun'] not in index". `autoreload` vuelve
# a leer el código en cada ejecución de celda.
# (Si algo se comporta de forma rara igualmente, reinicia el kernel: autoreload
#  no rehace objetos que ya estén creados en memoria.)
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 50)

raíz del proyecto: /Users/esteban/Documents/ACADEMICO/DSI/Radiacion_Solar/notebooks


In [2]:
from urbano import config as C
from urbano.clima import agregacion as G
from urbano.clima import variables as V

ImportError: cannot import name 'config' from 'urbano' (unknown location)

## 1. Las variables: unidad, altura y tipo

### ¿A qué altura está la temperatura? **A 2 m.**

Las variables meteorológicas del NSRDB no las mide el satélite: vienen de la
reanálisis **MERRA-2**. En el catálogo oficial de NREL
(`nsrdb/config/nsrdb_vars.csv`) cada una declara su dataset de origen:

| columna | dataset MERRA-2 | altura |
|---|---|---|
| `temperature` | `T2M` | **2 m**, corregida por elevación |
| `wind_speed`, `wind_direction` | `U2M` / `V2M` | **2 m** — *no* 10 m |
| `relative_humidity`, `dew_point` | derivadas de `T2M`+`QV2M`+`PS` | **2 m** |
| `pressure` | `PS` | superficie |
| `precipitable_water`, `ozone`, aerosoles | — | columna atmosférica |
| `ghi`, `dni`, `dhi` | GOES + PSM v4 | plano horizontal en superficie |

Dos consecuencias prácticas:

1. `temperature` **sí** es comparable con la temperatura de abrigo de una
   estación meteorológica (que también es a ~2 m).
2. `wind_speed` **no** lo es: la convención OMM para viento es **10 m**. Para
   compararlo con una estación estándar hay que extrapolar con perfil
   logarítmico, no usarlo tal cual.

In [ ]:
V.tabla()

In [ ]:
# Consultar una variable concreta.
v = V.CATALOGO["temperature"]
print(f"{v.nombre}: {v.descripcion}")
print(f"  unidad: {v.unidad} · altura: {v.altura} · tipo: {v.tipo}")

print("\nContinuas :", len(V.CONTINUAS))
print("Circulares:", V.CIRCULARES, "-> media vectorial, no aritmética")
print("Categóricas:", V.CATEGORICAS, "-> se rechazan en las agregaciones")
print("\nConjunto por omisión:", V.POR_OMISION)

In [ ]:
# Las categóricas se rechazan con un error explícito, no se promedian en
# silencio: `cloud_type` es un código, no una cantidad.
for mala in [["temperature", "cloud_type"], ["temperatura"]]:
    try:
        V.validar(mala)
    except ValueError as e:
        print("·", str(e)[:110], "…")

## 2. Serie horaria: la media ponderada

Tres decisiones que cambian el resultado:

1. **Ponderación por `peso`**, no media simple: el nodo que cubre el centro de
   una ciudad pesa más que el que apenas roza un borde.
2. **Renormalización ante NaN**: si un nodo no tiene dato en una hora, los pesos
   se reparten entre los presentes, en vez de sesgar la media hacia abajo.
3. **`wind_direction` se promedia en círculo**: promediar 350° y 10°
   aritméticamente daría 180°, el rumbo exactamente contrario.

In [ ]:
# Ejemplo mínimo: 3 ciudades, 2 variables, 1 año.
h = G.serie_horaria(["temperature", "relative_humidity"], top=3, anios=[2024])
print(h.shape)
h.head()

In [ ]:
# Las 12 ciudades mayores, 5 años, con el conjunto de variables por omisión.
horaria = G.serie_horaria(V.POR_OMISION, top=12,
                          anios=[2020, 2021, 2022, 2023, 2024])
print(f"{len(horaria):,} filas · {horaria['ciudad'].nunique()} ciudades")
print(f"{horaria['datetime_utc'].min()} → {horaria['datetime_utc'].max()} (UTC)")
horaria.head(3)

### El huso horario importa

El dataset viene en **UTC**. Si el día se cortara en UTC, el corte caería a las
18:00 hora local: partiría la tarde en dos y mezclaría el máximo de un día con el
del siguiente. Por eso el día se define en **hora local**.

Y no todo Tamaulipas está en el mismo huso: los **10 municipios de la franja
fronteriza** (Ley de los Husos Horarios, DOF 2022-10-28) conservaron el horario
de verano cuando el resto del país lo eliminó.

In [ ]:
horaria.groupby(["tz", "ciudad"]).size().rename("horas").reset_index()

In [ ]:
# El ciclo diario medio sale bien solo porque se agrupa por HORA LOCAL.
(horaria.assign(hora=horaria["hora_local"].dt.hour)
        .groupby(["ciudad", "hora"])["temperature"].mean()
        .unstack("ciudad").round(1))

## 3. Resumen diario: media, mínimo y máximo

`resumen_diario()` devuelve, por ciudad y **día local**, la media / mínimo /
máximo de cada variable continua, más `n_horas`.

`n_horas` no es decorativo: los días de cambio de horario traen 23 o 25 horas, y
el primero y el último del periodo vienen truncados. Filtra `n_horas == 24`
cuando compares días entre sí.

In [ ]:
diario = G.resumen_diario(horaria, V.POR_OMISION)
print(diario.shape)
print("columnas:", list(diario.columns))
diario.head()

In [ ]:
# `wind_direction` solo trae media (circular): su mín/máx no significan nada.
[c for c in diario.columns if c.startswith("wind_direction")]

In [ ]:
diario["n_horas"].value_counts().rename("días").to_frame()

In [ ]:
# Climatología por ciudad, ya sobre días completos.
(diario[diario["n_horas"] == 24]
 .groupby("ciudad")
 .agg(t_media=("temperature_media", "mean"),
      t_min_abs=("temperature_min", "min"),
      t_max_abs=("temperature_max", "max"),
      hr_media=("relative_humidity_media", "mean"),
      ghi_media=("ghi_media", "mean"))
 .round(2).sort_values("t_media", ascending=False))

## 4. Promedio ponderado vs. promedio simple

`ponderar=False` calcula el **promedio simple**: todos los nodos cuentan igual.
Está implementado como un caso particular del ponderado (pesos uniformes `1/n`),
así que hereda el mismo trato de los NaN y la misma media circular; lo único que
cambia es el vector de pesos.

- **Ponderado** (por omisión): representa mejor a la ciudad.
- **Simple**: más fácil de explicar ("el promedio de los N nodos") y no depende
  de la geometría de las manchas.

La brecha entre ambos **no** depende de cuántos nodos tenga la ciudad, sino de lo
**desigual** que sea el reparto de pesos.

In [ ]:
# San Fernando: 5 nodos, uno con el 64 % del peso. Es donde más se separan,
# más que Reynosa, cuyos 25 nodos ya reparten de forma casi uniforme.
ponderado = G.serie_horaria(["temperature"], cvegeos=["280350001"], anios=[2024])
simple    = G.serie_horaria(["temperature"], cvegeos=["280350001"], anios=[2024],
                            ponderar=False)

cmp = pd.DataFrame({"ponderado": ponderado["temperature"].to_numpy(),
                    "simple":    simple["temperature"].to_numpy()})
cmp["diferencia"] = cmp["ponderado"] - cmp["simple"]
cmp.describe().round(3)

In [ ]:
# Los pesos que usa cada modo, lado a lado.
pesos_sf = G.cargar_pesos(cvegeos=["280350001"])
comparacion = pesos_sf[["nodo_id", "peso"]].rename(columns={"peso": "ponderado"})
comparacion["simple"] = G.pesos_uniformes(pesos_sf)["peso"].to_numpy()
comparacion.sort_values("ponderado", ascending=False).round(4)

In [ ]:
# La brecha crece con lo desigual del reparto (`peso_max`), no con `n_nodos`.
brechas = []
for cve in diario["cvegeo"].unique():
    a = G.serie_horaria(["temperature"], cvegeos=[cve], anios=[2024])
    b = G.serie_horaria(["temperature"], cvegeos=[cve], anios=[2024],
                        ponderar=False)
    brechas.append({
        "ciudad": a["ciudad"].iloc[0],
        "n_nodos": int(a["n_nodos_ciudad"].iloc[0]),
        "peso_max": G.cargar_pesos(cvegeos=[cve])["peso"].max(),
        "dif_max_abs": (a["temperature"] - b["temperature"]).abs().max(),
    })
pd.DataFrame(brechas).sort_values("dif_max_abs", ascending=False).round(3)

## 5. Advertencias al comparar ciudades

1. De las 12, solo **4 tienen cobertura `alta`**. En las de cobertura `media` la
   serie está dominada por celdas que son mayoritariamente campo: una diferencia
   de décimas de grado entre dos de ellas no es una diferencia entre ciudades.
2. Ninguna serie capta la **isla de calor urbana**. MERRA-2 tiene ~50 km de
   resolución nativa y no modela el efecto urbano: `temperature` es la
   temperatura regional a 2 m interpolada a la celda, no la de la calle.
3. Para cruzar dos ciudades hora a hora, usa `datetime_utc`, **nunca**
   `hora_local`: hay dos husos en el estado.

In [ ]:
diario.groupby(["calidad", "ciudad"]).size().rename("días").reset_index()

In [ ]:
# Cruce correcto entre dos ciudades de husos distintos.
par = horaria[horaria["ciudad"].isin(["Reynosa", "Tampico"])]
(par.pivot_table(index="datetime_utc", columns="ciudad", values="temperature")
    .assign(diferencia=lambda d: d["Reynosa"] - d["Tampico"])
    .describe().round(2))

## Qué sigue

**03 · consultas** — la interfaz que envuelve todo esto: pedir por nombre o
clave, acotar fechas y cambiar de ámbito sin recalcular pesos a mano.